In [ ]:
import numpy as np
from typing import Iterable, List, Tuple, Union
from netCDF4 import Dataset
import datetime

try:
    from pyproj import CRS, Transformer
    _HAS_PYPROJ = True
except Exception:
    _HAS_PYPROJ = False

In [ ]:
R  = 6400

def distance(lat1, lon1, lat2, lon2):
    lat_av = np.deg2rad(lat1 - lat2)/2
    lon_av = np.deg2rad(lon1 - lon2)/2
    dd = np.sin(lat_av)**2 + np.cos(np.deg2rad(lat2)) * np.cos(np.deg2rad(lat1)) * np.sin(lon_av)**2
    dd = 2*R*np.arcsin(np.sqrt(dd))
    return dd

In [ ]:
def _coriolis_f_at_75deg() -> float:
    """
    Параметр Кориолиса f на широте 75° (с^-1).
    """
    omega = 7.2921159e-5  # скор. вращения Земли, с^-1
    phi = np.deg2rad(75.0)
    return 2.0 * omega * np.sin(phi)

In [ ]:
def _drag_coefficient_large_pond(U10: float) -> float:
    """
    Коэффициент трения воздуха C_d по Large & Pond (1981) для 10-метрового ветра.
    Возвращает безразмерную величину.
    """
    U = np.maximum(U10, 0.0)
    if U < 3.0:
        return 0.0010
    elif U <= 20.0:
        return (0.63 + 0.066 * U) * 1e-3
    else:
        return 0.0026

In [ ]:
def _wind_stress(u10: float, v10: float, rho_air: float = 1.225) -> Tuple[float, float]:
    """
    Ветровой стресс τ = ρ_air * C_d * |U| * (u, v), Н/м^2.
    Возвращает (tau_x, tau_y).
    """
    U = np.hypot(u10, v10)
    Cd = _drag_coefficient_large_pond(U)
    factor = rho_air * Cd * U
    return factor * u10, factor * v10

In [ ]:
def ekman_displacement_km(
    coords: List[List[float]],
    winds: List[List[float]],
    dt_seconds: float,
    H_m: float = 20.0,
    rho_w: float = 1025.0,
    use_pyproj: bool = True,
    return_dxdy = False,
) -> List[List[float]]:
    """
    Рассчитывает экмановское смещение для набора точек за интервал времени dt_seconds.
    Вход:
        coords: [[lat_deg, lon_deg], ...]
        winds:  [[u_wind_ms, v_wind_ms], ...]
    Выход:
        [[dx_km, dy_km, lat_new, lon_new], ...]
    """
    if dt_seconds <= 0:
        raise ValueError("dt_seconds должен быть > 0")

    if len(coords) != len(winds):
        raise ValueError("Длина coords и winds должна совпадать")

    # Кориолис на 75° — общая константа для всех точек
    f = _coriolis_f_at_75deg()

    results: List[List[float]] = []

    # Проверка наличия pyproj
    use_proj = use_pyproj and _HAS_PYPROJ
    if use_proj:
        from pyproj import CRS, Transformer

    for (lat_deg, lon_deg), (u_wind_ms, v_wind_ms) in zip(coords, winds):
            # 1) Ветровой стресс
            tau_x, tau_y = _wind_stress(u_wind_ms, v_wind_ms)

            # 2) Экмановская скорость (усреднённая по H): U_E = (1/(rho_w f H)) * (τ × k)
            coeff = 1.0 / (rho_w * f * H_m)
            Ue_x = coeff * (tau_y)   # м/с
            Ue_y = coeff * (-tau_x)  # м/с

            # 3) Смещение за dt
            dx_m = Ue_x * dt_seconds
            dy_m = Ue_y * dt_seconds
            dx_km = dx_m / 1000.0
            dy_km = dy_m / 1000.0

            # 4) Новые координаты
            if use_proj:
                crs_geodetic = CRS.from_epsg(4326)
                crs_local = CRS.from_proj4(
                    f"+proj=aeqd +lat_0={lat_deg} +lon_0={lon_deg} +datum=WGS84 +units=m +no_defs"
                )
                to_local = Transformer.from_crs(crs_geodetic, crs_local, always_xy=True)
                to_geo = Transformer.from_crs(crs_local, crs_geodetic, always_xy=True)

                x0, y0 = to_local.transform(lon_deg, lat_deg)
                x1, y1 = x0 + dx_m, y0 + dy_m
                lon_new, lat_new = to_geo.transform(x1, y1)
                lon_new = (lon_new + 180.0) % 360.0 - 180.0
            else:
                R_E = 6371000.0
                dlat = (dy_m / R_E) * (180.0 / np.pi)
                dlon = (dx_m / (R_E * np.cos(np.deg2rad(lat_deg)))) * (180.0 / np.pi)
                lat_new = lat_deg + dlat
                lon_new = (lon_deg + dlon + 180.0) % 360.0 - 180.0

            if return_dxdy:
                results.append([float(dx_km), float(dy_km), float(lat_new), float(lon_new)])
            else:
                 results.append([float(lat_new), float(lon_new)])

    return results

In [ ]:
def get_wind_components_at_points(
    lat_grid: np.ndarray,
    lon_grid: np.ndarray,
    u: np.ndarray,
    v: np.ndarray,
    points: Iterable[Tuple[float, float]],
    wrap_longitude: bool = True,
    outside: str = "nan",
) -> np.ndarray:
    """
    Возвращает компоненты скорости ветра (u, v) для набора точек (широта, долгота),
    интерполируя поле u и v на регулярной сетке (meshgrid) билинейно.

    Параметры:
      - lat_grid: 2D массив широт (как из np.meshgrid), совпадает по форме с lon_grid, u, v.
      - lon_grid: 2D массив долгот (как из np.meshgrid), совпадает по форме с lat_grid, u, v.
      - u: 2D массив компоненты ветра по долготе (ось x), та же форма.
      - v: 2D массив компоненты ветра по широте (ось y), та же форма.
      - points: Iterable из кортежей (широта, долгота) — порядок обязательный: (lat, lon).
      - wrap_longitude: если True — долготы входных точек приводятся к диапазону сетки
                        (0..360 или -180..180).
      - outside: "nan" — вернуть NaN для точки вне диапазона; "clamp" — зажать к границе;
                 "raise" — вызвать исключение.

    Возвращает:
      - np.ndarray формы (N, 2), где N — число точек; столбцы: [u_interp, v_interp].

    Примечания:
      - Используется билинейная интерполяция по четырем соседним узлам.
      - Предполагается регулярная монотонная сетка по широте и долготе.
    """
    # Проверка формы
    if lat_grid.shape != lon_grid.shape or lat_grid.shape != u.shape or lat_grid.shape != v.shape:
        raise ValueError("lat_grid, lon_grid, u и v должны иметь одинаковую 2D-форму.")

    # Оси широты/долготы из meshgrid (строки — широты, столбцы — долготы)
    lat_axis = lat_grid[:, 0]
    lon_axis = lon_grid[0, :]

    def axis_bounds(axis: np.ndarray):
        asc = axis[0] < axis[-1]
        return (min(axis[0], axis[-1]), max(axis[0], axis[-1]), asc)

    lat_min, lat_max, lat_asc = axis_bounds(lat_axis)
    lon_min, lon_max, lon_asc = axis_bounds(lon_axis)

    def normalize_lon(lon: float) -> float:
        if not wrap_longitude:
            return lon
        # Типичная глобальная сетка: либо 0..360, либо -180..180
        if lon_min >= 0 and lon_max <= 360:
            return lon % 360.0
        else:
            return ((lon + 180.0) % 360.0) - 180.0

    def is_outside(x: float, xmin: float, xmax: float) -> bool:
        return (x < xmin) or (x > xmax)

    def bracket_and_weight(axis: np.ndarray, x: float, asc: bool):
        """
        Возвращает индексы (i0, i1) и вес w в [0,1], такие что
        x = axis[i0] + w * (axis[i1] - axis[i0]).
        """
        n = axis.size
        if n < 2:
            raise ValueError("Длина оси должна быть >= 2.")
        if asc:
            i = np.searchsorted(axis, x)
            if i == 0:
                i0, i1 = 0, 1
            elif i >= n:
                i0, i1 = n - 2, n - 1
            else:
                i0, i1 = i - 1, i
        else:
            axis_rev = axis[::-1]
            i = np.searchsorted(axis_rev, x)
            if i == 0:
                i0_rev, i1_rev = 0, 1
            elif i >= n:
                i0_rev, i1_rev = n - 2, n - 1
            else:
                i0_rev, i1_rev = i - 1, i
            i0 = n - 1 - i1_rev
            i1 = n - 1 - i0_rev

        x0, x1 = axis[i0], axis[i1]
        w = 0.0 if x1 == x0 else (x - x0) / (x1 - x0)
        return i0, i1, float(w)

    # Преобразуем точки в список один раз
    pts: List[Tuple[float, float]] = list(points)
    N = len(pts)
    comps = np.empty((N, 2), dtype=float)

    for k, (lat_p, lon_p) in enumerate(pts):
        lon_p = normalize_lon(float(lon_p))
        lat_p = float(lat_p)

        out_lat = is_outside(lat_p, lat_min, lat_max)
        out_lon = is_outside(lon_p, lon_min, lon_max)

        if (out_lat or out_lon):
            if outside == "nan":
                comps[k, :] = np.nan
                continue
            elif outside == "raise":
                raise ValueError(f"Точка вне диапазона сетки: lat={lat_p}, lon={lon_p}")
            elif outside == "clamp":
                lat_p = min(max(lat_p, lat_min), lat_max)
                lon_p = min(max(lon_p, lon_min), lon_max)
            else:
                raise ValueError("Параметр 'outside' должен быть 'nan', 'clamp' или 'raise'.")

        i0_lat, i1_lat, t = bracket_and_weight(lat_axis, lat_p, lat_asc)
        i0_lon, i1_lon, s = bracket_and_weight(lon_axis, lon_p, lon_asc)

        # Билинейная интерполяция
        u00 = u[i0_lat, i0_lon]; u01 = u[i0_lat, i1_lon]
        u10 = u[i1_lat, i0_lon]; u11 = u[i1_lat, i1_lon]
        v00 = v[i0_lat, i0_lon]; v01 = v[i0_lat, i1_lon]
        v10 = v[i1_lat, i0_lon]; v11 = v[i1_lat, i1_lon]

        u_interp = (1 - s) * (1 - t) * u00 + s * (1 - t) * u01 + (1 - s) * t * u10 + s * t * u11
        v_interp = (1 - s) * (1 - t) * v00 + s * (1 - t) * v01 + (1 - s) * t * v10 + s * t * v11

        comps[k, 0] = float(u_interp)
        comps[k, 1] = float(v_interp)

    return comps

In [ ]:
lon_min_era5, lon_max_era5, lat_min_era5, lat_max_era5 = 35, 105, 66, 82
lat1, lat2 = (90-lat_max_era5)*4, (90-lat_min_era5)*4+1
lon1, lon2 = lon_min_era5*4, lon_max_era5*4+1

In [ ]:
month = '2024-09'
file = f'/mnt/hippocamp/DATA/ERA5/w10/era5_uv10m_{month}.nc'
data = Dataset(file, 'r')

tt = np.asarray(data.variables['valid_time'])
time = np.asarray([datetime.datetime(1970, 1, 1, 0, 0, 0) + datetime.timedelta(seconds=int(t)) for t in tt])
u10 = data.variables['u10'][:,lat1:lat2,lon1:lon2]
v10 = data.variables['v10'][:,lat1:lat2,lon1:lon2]

longitude = np.array(data.variables['longitude'][lon1:lon2])
latitude = np.array(data.variables['latitude'][lat1:lat2])
lon_grid, lat_grid = np.meshgrid(longitude, latitude)

data.close()

In [ ]:
date_start = datetime.datetime(2024, 9, 6, 12, 0)
date_finish = datetime.datetime(2024, 9, 9, 12, 0)
points = [
    (76.250, 72),
    (76.250, 74),
    (76.250, 76),
    (76.250, 78),
    (76.250, 80),
]
t_start = np.where(time == date_start)[0][0]
t_finish = np.where(time == date_finish)[0][0]

In [ ]:
dt_seconds = 3600
points_coords = []
points_coords.append(points)

for t in range(t_start, t_finish):
    uv = get_wind_components_at_points(lat_grid, lon_grid, u10[t], v10[t], points_coords[-1], wrap_longitude=True, outside="clamp")
    results = ekman_displacement_km(coords=points_coords[-1], winds=uv, dt_seconds=dt_seconds, H_m=20.0, rho_w=1025.0, use_pyproj=True, return_dxdy=False)
    points_coords.append(results)

In [ ]:
len(points_coords)

In [ ]:
points_coords[0], points_coords[-1]

In [ ]:
for i in range(len(points_coords[0])):
    print(distance(points_coords[0][i][0], points_coords[0][i][1], points_coords[-1][i][0], points_coords[-1][i][1]))

In [ ]:
results

In [ ]:
u10_0610 = []
v10_0610 = []
for day in range(6, 11):
    u10_0610.append(np.mean(u10[np.where(time == datetime.datetime(2024, 9, day, 0, 0))[0][0]:np.where(time == datetime.datetime(2024, 9, day+1, 0, 0))[0][0]], axis=0))
    v10_0610.append(np.mean(v10[np.where(time == datetime.datetime(2024, 9, day, 0, 0))[0][0]:np.where(time == datetime.datetime(2024, 9, day+1, 0, 0))[0][0]], axis=0))

In [ ]:
dt_seconds = 24 * 3600
points_coords = []
points = [
    (76.250, 72),
    (76.250, 74),
    (76.250, 76),
    (76.250, 78),
    (76.250, 80),
]
points_coords.append(points)

for t in range(5):
    uv = get_wind_components_at_points(lat_grid, lon_grid, u10_0610[t], v10_0610[t], points_coords[-1], wrap_longitude=True, outside="clamp")
    print(uv)
    results = ekman_displacement_km(coords=points_coords[-1], winds=uv, dt_seconds=dt_seconds, H_m=20.0, rho_w=1025.0, use_pyproj=True, return_dxdy=False)
    points_coords.append(results)

In [ ]:
points_coords[0], points_coords[-1]

In [ ]:
t = 168
u = u10[t]
v = v10[t]

points = [
        [76, 70],    # форма: (широта, долгота)
        [76, 80]
    ]

uv = get_wind_components_at_points(lat_grid, lon_grid, u, v, points, wrap_longitude=True, outside="clamp")
print(uv)  # форма: (N, 2), столбцы [u, v]

In [ ]:
# Входные данные: три точки и соответствующие ветровые скорости
coords: List[List[float]] = [
    [75.0, 40.0],    # [lat, lon]
    [74.5, 39.5],
    [73.8, 41.2],
]

winds: List[List[float]] = [
    [10.0, -3.0],    # [u_wind_ms, v_wind_ms]
    [8.5, 1.2],
    [12.0, -5.5],
]

# Длительность шага интегрирования (в секундах)
dt_seconds: float = 3600.0  # 1 час

# Вызов функции
results = ekman_displacement_km(
    coords=coords,
    winds=winds,
    dt_seconds=dt_seconds,
    H_m=20.0,
    rho_w=1025.0,
    use_pyproj=True,
    return_dxdy=True
)

# Вывод результатов
for i, (dx_km, dy_km, lat_new, lon_new) in enumerate(results, start=1):
    lat0, lon0 = coords[i - 1]
    u, v = winds[i - 1]
    print(f"Точка {i}:")
    print(f"  Исходные координаты: lat={lat0:.4f}, lon={lon0:.4f}")
    print(f"  Ветер (u, v) м/с: ({u:.2f}, {v:.2f})")
    print(f"  Смещение: dx={dx_km:.4f} км, dy={dy_km:.4f} км")
    print(f"  Новые координаты: lat={lat_new:.4f}, lon={lon_new:.4f}")
    print("-" * 40)